# Detecting registration errors in OMR answer sheets

A candidate skips a bubble row. Every answer after that sits one row too low.
All of them are correct. All of them are marked wrong.

Correcting that is easy. Correcting it *without* handing marks to people who
did not earn them is the whole problem, and it is what this notebook shows.
Runtime is about a minute.

Everything below reads from results committed in the repository, so the numbers
here are the numbers in the report.


## Setup

Runs in Colab or locally. In Colab it clones the repository; locally it uses
the files already present.


In [ ]:
# Opening this notebook from GitHub in Colab does not bring the code with it,
# so fetch the repository first.
import os, sys, time

REPO = "https://github.com/Enayat-Hassani/omr-shift-detection"

if not os.path.exists("omr_shift.py"):
    name = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(name):
        os.system(f"git clone --depth 1 {REPO} 2>/dev/null")
    if os.path.isdir(name):
        os.chdir(name)

if not os.path.exists("omr_shift.py"):
    raise SystemExit(
        "omr_shift.py not found. Either set REPO above to the repository URL, "
        "or upload the .py files with:  from google.colab import files; files.upload()")

sys.path.insert(0, ".")
from omr_shift import *
print("ready:", os.getcwd())


## 1. The sheet that started it

46 maths questions. The candidate scored 7. Their other subjects were
reportedly fine, which is why somebody suspected a shift.


In [ ]:
sheet = ResponseSheet.from_file(CASE_SHEET)
print("key    :", "".join(sheet.key))
print("student:", "".join(m or '.' for m in sheet.marks))
print(f"\nscore as marked: {sheet.raw_score()} out of {sheet.n_questions}")
print(f"random guessing would give about {sheet.n_questions/4:.0f}")


## 2. Why the obvious approach does not work

The generous fix is to find the longest run of the candidate's answers that
appears in the key in the same order, and award that. No settings, handles
shifts automatically.

Watch what it does to a sheet filled in at random.


In [ ]:
import random
def lcs(a, b):
    prev = [0]*(len(b)+1)
    for x in a:
        cur = [0]*(len(b)+1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j-1]+1 if x == y else max(prev[j], cur[j-1])
        prev = cur
    return prev[-1]

rng = random.Random(1)
strict   = sheet.raw_score()
generous = lcs(sheet.marks, sheet.key)
rand = [lcs([rng.choice("ABCD") for _ in range(46)], sheet.key) for _ in range(30)]

print(f"this sheet, marked strictly      : {strict} / 46")
print(f"this sheet, marked generously    : {generous} / 46")
print(f"pure guesswork, marked generously: {sum(rand)/len(rand):.1f} / 46  (chance alone gives 11.5)")
print()
print("A sheet with no knowledge on it scores in the twenties. Any rule\n"
      "permissive enough to rescue a real error is permissive enough to\n"
      "manufacture one. That is the problem this project is about.")


## 3. What the detector says about this sheet

Five checks. Every one has to pass before a single mark moves.


In [ ]:
t = time.time()
adj = Adjudicator(sheet, AdjudicationConfig()).run(n_permutations=999, verbose=False)
print(f"({time.time()-t:.1f} seconds)\n")
print(f"VERDICT: {adj.verdict}\n")
print(f"score as marked : {adj.raw_score} / 46")
print(f"score after     : {adj.adjudicated_score} / 46\n")
for name, g in adj.gates.items():
    print(f"  [{'pass' if g['passed'] else 'FAIL'}] {name}")
print()
w = adj.calibration["scan_window"]
if w:
    print("Best run of correct answers at any shifted position:")
    print(f"  questions {w['q_start']} to {w['q_end']}, shifted by {w['offset']:+d}, "
          f"{w['n_correct']} right out of {w['n_items']}")
print(f"Sheets known to contain no shift do that well or better "
      f"{adj.calibration['p_value']:.0%} of the time.")


## 4. Does it catch a real one?

A negative result means nothing unless the detector works. Same answer key, a
candidate who knows 85% of the material, one bubble row skipped.


In [ ]:
ctrl = demo_planted_shift(AdjudicationConfig(), '.')
print(f"VERDICT: {ctrl.verdict}\n")
print(f"score as marked : {ctrl.raw_score} / 46")
print(f"score after     : {ctrl.adjudicated_score} / 46")
gained = sum(1 for r in ctrl.item_ledger if r['change'] == 'GAIN')
lost   = sum(1 for r in ctrl.item_ledger if r['change'] == 'LOSS')
print(f"marks gained {gained}, marks lost {lost}   (losses are counted too)")


## 5. One question at a time

Even after every check passes, a question is only re-read if the model is at
least 99% sure which row it belongs to. Questions near the edge of the shift
are left alone. Marks that would be *lost* by re-reading are subtracted.


In [ ]:
for r in ctrl.item_ledger[13:22]:
    mark = 'yes' if r['final_correct'] else 'no '
    print(f"  Q{r['question']:>2}  key {r['key']}  confidence {r['map_posterior']:.3f}  "
          f"correct now: {mark}  {r['reason'][:52]}")


## 6. Five detectors on identical sheets

Ten candidate models, each built to break a different assumption. Read from
the committed benchmark; the run itself takes minutes.


In [ ]:
import json

# The provenance record wraps the rows -- see section 8.
R = json.load(open("results/benchmark.json"))["results"]
order = ["no-op (never correct)", "brute-force shift", "LCS (maximally generous)",
         "fixed-cost DP alignment", "gated pair-HMM (reference)"]
noop = sum(r["marks_wrongly_withheld"] for r in R
           if r["detector"] == order[0]) / len([r for r in R if r["detector"] == order[0]])

print(f"{'detector':<28}{'worst FPR':>11}{'awarded':>10}{'recovery':>10}{'Brier':>9}")
print("-" * 68)
for d in order:
    rs = [r for r in R if r["detector"] == d]
    br = [r["brier"] for r in rs if r["brier"] is not None]
    hold = sum(r["marks_wrongly_withheld"] for r in rs) / len(rs)
    print(f"{d:<28}{max(r['fpr'] for r in rs):>11.2f}"
          f"{sum(r['marks_wrongly_awarded'] for r in rs)/len(rs):>10.2f}"
          f"{(noop-hold)/noop:>9.0%}"
          f"{(f'{sum(br)/len(br):.3f}' if br else 'none'):>9}")

print()
print("worst FPR is the highest false-alarm rate across the ten candidate models.")
print("awarded is marks given that were not earned. recovery is the share of marks")
print("lost to an error that the detector returns. Brier measures whether the")
print("reported confidence is trustworthy; lower is better, and it is undefined")
print("for detectors that report no confidence at all.")


## 7. Where the rivals break, and why this one does not

A single worst-case number hides the interesting part. Broken out by candidate
model, the failures are not scattered — they land on the four models that
produce **runs** of the same option.


In [ ]:
import re
txt = open("results/benchmark.txt").read()
blocks = re.split(r"^GENERATOR:\s*(\S+)", txt, flags=re.M)
fpr = {}
for i in range(1, len(blocks), 2):
    gen, body = blocks[i], blocks[i+1]
    for m in re.finditer(r"^\s{2}(\S.*?)\s{2,}([\d.]+)\s+\(", body, re.M):
        name = m.group(1).strip()
        if name != "detector":
            fpr.setdefault(gen, {})[name] = float(m.group(2))

show = ["gated pair-HMM (reference)", "brute-force shift",
        "fixed-cost DP alignment", "LCS (maximally generous)"]
print(f"{'candidate model':<24}" + "".join(f"{s.split()[0][:9]:>11}" for s in show))
print("-" * 68)
for gen in sorted(fpr, key=lambda g: -fpr[g][show[2]]):
    print(f"{gen:<24}" + "".join(f"{fpr[gen][s]:>11.2f}" for s in show))

print()
print("The gated model is at 0.00 everywhere. Fixed-cost alignment accepts EVERY")
print("clean sheet from the adversarial model, and fails on exactly the four that")
print("produce long runs of one option -- a streaky candidate manufactures blocks")
print("that look displaced and correct. The gate survives that because two of its")
print("three null models resample the candidate's OWN answers, so their streaks")
print("appear in the null too and stop being surprising.")


## 8. Can you trust these numbers?

A fair question to ask of any repository, so the project answers it
mechanically rather than by assertion.

Every results file records a fingerprint of the code that produced it, and
regenerating from unchanged code reproduces the file byte for byte — so an
empty `git diff` is itself the check. The fingerprint is a hash of the source
actually loaded, not the commit: a commit id changes whenever history is
rewritten, without a line of code changing. The commit below is recorded as
context and may well no longer exist; the fingerprint is the evidence.


In [ ]:
block = open("results/benchmark.txt").read().split("PROVENANCE")[-1]
print("PROVENANCE" + block.strip()[:520])


Every number published in the README and the report is registered
against the file it came from, and checked. When this last ran it found ten
stale figures.


In [ ]:
!python3 tools/check_figures.py | tail -8


The fairness rules are asserted against randomly generated sheets rather
than trusted. This suite is what caught a bug where one physical mark could be
credited to two different questions.


In [ ]:
!python3 tests/test_invariants.py 2>&1 | tail -4


## 9. Screening a whole sitting

Judging one disputed sheet is one problem. Screening four thousand and acting
on whatever passes is a different one: a per-sheet rate that is negligible once
is not negligible four thousand times.

The measured answer is that it works, and that it needs a long enough paper.


In [ ]:
print(open("results/cohort_screen.txt").read().split("PROVENANCE")[0].strip())


## 10. Figures

Two of the committed figures. The first shows the null calibration for the
case sheet: the observed evidence sits inside the bulk of what error-free
sheets produce. The second shows the positive control, where a planted error
produces a clean change in registration with genuine uncertainty at its edges.


In [ ]:
# IPython is always present in Colab; locally it may not be.
try:
    from IPython.display import Image, display
    _inline = True
except ImportError:
    _inline = False

for path, caption in [
    ("results/figures/4_null_calibration.png",
     "Case sheet: observed evidence against three null models"),
    ("results/figures/positive_control/1_displacement_posterior.png",
     "Positive control: per-question posterior over displacement"),
]:
    print(caption)
    if _inline:
        display(Image(path))
    else:
        print(f"  -> {path}\n")


## Reading further

All of these are in the repository and need no execution.

| | |
|---|---|
| `REPORT.md` | Problem, models evaluated, comparison, recommended configuration, safeguards |
| `ASSUMPTIONS.md` | Six assumptions made during design, and what measurement said about each |
| `CASE_REPORT.md` | Full analysis of the sheet in section 1 |
| `results/case_default_prior.txt` | Every question of that sheet, with the reasoning |
| `results/benchmark.txt` | Per-generator benchmark tables |
| `REPRODUCE.md` | The commands that regenerate every published number |

The benchmarks are `benchmark/omrbench.py` (minutes) and
`benchmark/large_synthetic.py` (about 1.6 hours for 9,984 sheets), which is why
their output is committed here.
